# Workshop: Streaming & Auto Loader

## Scenario

New order files arrive continuously in a landing zone on cloud storage.  
Your goal is to build a streaming ingestion pipeline that picks up each file **exactly once**, lands it in the Bronze layer, and tracks progress so restarts are safe.

You will compare two ingestion approaches (**COPY INTO** vs **Auto Loader**) and then extend the pipeline with metadata enrichment, schema evolution handling, and — as a bonus — change-based incremental ETL.


## Key Concepts

### COPY INTO
A SQL command that loads files from a path into a Delta table.  
It is **idempotent**: Databricks tracks which files have already been loaded and skips them on re-runs. Simple, but limited to thousands of files.

```sql
COPY INTO my_table
FROM '/path/to/files'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' => 'true')
```

### Auto Loader (`cloudFiles`)
A Structured Streaming source that ingests files incrementally from cloud storage.  
Uses a **checkpoint directory** to remember which files have been processed — safe to restart.  
Scales to millions of files (uses cloud file-notification APIs instead of directory listing).

```python
spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", schema_path)
  .option("cloudFiles.inferColumnTypes", "true")
  .load(landing_path)
```

### `trigger(availableNow=True)`
Tells the stream to process **all currently available files** and then stop automatically.  
Equivalent to a batch run, but uses all the streaming guarantees (checkpoint, exactly-once). Ideal for scheduled incremental loads.

### Checkpoint
A directory that records the stream's progress (which files/offsets have been processed).  
If the cluster restarts, the stream resumes from the checkpoint — **no duplicates, no data loss**.

### Metadata columns
Auto Loader exposes hidden file-level metadata via the `_metadata` struct.  
Useful columns to add to every Bronze record:

| Column | Expression | Purpose |
|--------|-----------|---------|
| `_processing_time` | `current_timestamp()` | When the row was processed |
| `_source_file` | `col("_metadata.file_path")` | Which file the row came from |

### Schema Evolution — Rescued Data
When Auto Loader reads a file with more columns than the defined schema, those extra columns are **not dropped** — they are captured as a JSON string in `_rescued_data`.  
Enable with: `.option("cloudFiles.schemaEvolutionMode", "rescue")`


## Prerequisites

- Cluster is running and attached to the notebook
- Run the **Setup** cell first — it creates paths and drops leftover tables from previous runs
- Source files are in `dataset/orders/stream/` (3 JSON files)
- The setup cell also creates the `customers` Bronze table used in Task 8 (Bonus)


## Tasks Overview

| Task | Topic | What you will do |
|------|-------|-----------------|
| 1 | COPY INTO | Load JSON files from the landing zone into a Delta table using a SQL command |
| 2 | Idempotency | Re-run the same COPY INTO — verify that 0 new rows are added |
| 3 | Auto Loader — read | Configure a streaming `readStream` with `cloudFiles` format |
| 4 | Auto Loader — write | Write the stream to a Delta table with `trigger(availableNow=True)` |
| 5 | Incremental processing | Re-run the stream — checkpoint prevents reprocessing already-seen files |
| 6 | Metadata columns | Enrich each row with `_processing_time` and `_source_file` |
| 7 | Schema rescue | Use a partial schema + rescue mode — extra columns go to `_rescued_data` |
| **8** | **BONUS** — Stream-Static Join | Enrich the order stream with customer data from a static table |
| **9** | **BONUS** — Change Data Feed | Enable CDF, make changes, read only the delta with `table_changes()` |

> Tasks 8 and 9 are **optional** — attempt them if time allows after Task 7.


## Hints — Tasks 1–7

### Task 1: COPY INTO
Write a `COPY INTO` statement inside `spark.sql(f"...")`.  
You need three clauses:
- `COPY INTO {target_table}` — destination Delta table
- `FROM '{landing_path}'` — source path (use the variable)
- `FILEFORMAT = JSON` and `FORMAT_OPTIONS ('mergeSchema' => 'true')`

```sql
COPY INTO my_table
FROM '/landing/path'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' => 'true')
```

---

### Task 2: Idempotency
Copy the **exact same** COPY INTO statement from Task 1 and run it again.  
Databricks tracks loaded files internally — the row count must not change.  
Expected: `count_after_rerun == count_after_copy`

---

### Task 3: Auto Loader — configure readStream
Replace the `...` placeholder with a full `spark.readStream` chain:

```python
df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(landing_path)
)
```

Check: `df_stream.isStreaming` must be `True`.

---

### Task 4: Write Stream
Chain `.writeStream` off `df_stream`:

```python
query = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(al_target)
)
query.awaitTermination()
```

`awaitTermination()` blocks until the stream finishes processing all available files.

---

### Task 5: Incremental processing
Repeat **exactly** the same `readStream` + `writeStream` code from Tasks 3–4.  
Use the **same** `checkpoint_path` and `schema_path` — do not reset them.  
Because the checkpoint is intact, Auto Loader knows all three files were already processed → 0 new rows written.

---

### Task 6: Metadata columns
After the `spark.readStream...load(landing_path)` call, add `.withColumn(...)` twice:

```python
df_with_metadata = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", metadata_schema)
        .load(landing_path)
        .withColumn("_processing_time", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
)
```

`current_timestamp()` and `col` are already imported at the top of the cell.

---

### Task 7: Schema rescue
Use a **fixed schema** (provided as `partial_schema`) instead of `inferColumnTypes`.  
Add the rescue option:

```python
df_rescued = (
    spark.readStream
        .format("cloudFiles")
        .schema(partial_schema)
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", rescue_schema)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .load(landing_path)
)
```

Because `partial_schema` only declares `order_id` and `customer_id`, all other columns from the JSON files (quantity, total_amount, etc.) will be serialised into `_rescued_data`.


## Hints — Bonus Tasks 8–9

### Task 8: Stream-Static Join

#### What is a Stream-Static Join?
A **Stream-Static Join** combines a **streaming DataFrame** (rows arriving continuously) with a **static DataFrame** (a regular table read once per micro-batch).  
The static side is **re-read on every micro-batch** — so if the table changes between batches, the stream picks up the latest version automatically.  
This pattern is used to enrich a stream with reference data (e.g., customers, products, lookup tables).

```
Streaming orders  ──┐
                    ├── LEFT JOIN on customer_id ──> enriched stream
Static customers  ──┘
```

#### Code pattern

```python
# Static side — reads the full table as a batch DataFrame
customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

# Streaming side — reads a Delta table as a stream
orders_stream = (
    spark.readStream
        .format("delta")
        .table(target_table)        # target_table from Task 1
)

# Join — streaming left join static
enriched_stream = orders_stream.join(customers_df, on="customer_id", how="left")

# Write
(
    enriched_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", join_checkpoint)
        .trigger(availableNow=True)
        .toTable(join_target)
        .awaitTermination()
)
```

> Note: only `append` and `complete` output modes are supported for stream-static joins.

---

### Task 9: Change Data Feed (CDF)

#### What is CDF?
**Change Data Feed** records every row-level change made to a Delta table as a separate entry — inserts, updates (before and after), and deletes.  
Instead of re-reading the whole table, downstream pipelines read only what changed since the last run. Ideal for propagating changes through Medallion layers.

Each row in the change feed has extra columns:

| Column | Values | Meaning |
|--------|--------|---------|
| `_change_type` | `insert`, `update_preimage`, `update_postimage`, `delete` | What happened |
| `_commit_version` | integer | Delta table version of the change |
| `_commit_timestamp` | timestamp | When the change was committed |

For incremental ETL keep only **`insert`** and **`update_postimage`** (the new values after an update).

#### Step 1 — Create CDF-enabled table
```sql
CREATE TABLE my_table (
    order_id INT, customer_id INT, amount DOUBLE, status STRING
)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
```

#### Step 2 — Read only the changes since a given version
```python
df_changes = spark.sql(f"""
    SELECT * FROM table_changes('{cdf_source}', {initial_version + 1})
""")
```
`initial_version + 1` skips the initial bulk INSERT and reads only subsequent changes.

#### Step 3 — Filter to useful change types
```python
from pyspark.sql.functions import col

df_incremental = df_changes.filter(
    col("_change_type").isin("insert", "update_postimage")
)
```


## Summary

### COPY INTO vs Auto Loader vs CDF

| | COPY INTO | Auto Loader | CDF |
|---|-----------|-------------|-----|
| **What it reads** | Files from cloud storage | Files from cloud storage | Row-level changes in a Delta table |
| **API** | SQL command | `readStream` / `writeStream` | `table_changes()` or `readStream` with `readChangeFeed` |
| **File scale** | Thousands of files | Millions of files | N/A (table-based) |
| **Schema evolution** | Manual | Automatic (`rescue`, `addNewColumns`) | Follows source schema |
| **Progress tracking** | Internal SQL state | Checkpoint directory | Delta version number |
| **Typical use case** | Simple periodic batch load | Continuous / large-scale file ingestion | Propagating updates through Medallion layers |

### Exam Tips

- Auto Loader uses **`cloudFiles`** as the format name.
- `trigger(availableNow=True)` processes all pending files and **stops** — use it for scheduled jobs instead of a continuous stream.
- The **checkpoint** is what makes Auto Loader exactly-once: it records processed files so restarts are safe and duplicates are prevented.
- CDF captures `_change_type`: `insert`, `update_preimage` (old row), `update_postimage` (new row), `delete`. For incremental ETL keep `insert` + `update_postimage`.
- A **Stream-Static Join** re-reads the static side on every micro-batch — the static table always reflects its latest state.
